# q_simulator tutorial

`q_simulator` accepts Qiskit quantum circuits and returns a final statevector together with measurement probabilities or counts.

This tutorial creates a Bell state, simulates it with each available backend, and compares the results with Qiskit.

## Imports and circuit setup

In [6]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

import q_simulator as qs

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)

The circuit prepares the Bell state

$$\frac{|00\rangle + |11\rangle}{\sqrt{2}}.$$

The simulator configuration selects a backend and the requested number of shots.

In [7]:
shots = 2**12
methods = ["default", "einsum", "loop", "numba"]

results = {}
for method in methods:
    config = qs.Configuration(method=method, number_of_shots=shots)
    results[method] = qs.simulate(circuit.copy(), config)

for method, result in results.items():
    print(f"{method:>7}: {result.statevector}")

default: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
 einsum: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
   loop: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
  numba: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


The statevectors should agree across all methods.

In [ ]:
reference = Statevector.from_instruction(circuit).data

for method, result in results.items():
    assert np.allclose(result.statevector, reference)

print("All simulation methods match the Qiskit reference statevector.")

All simulation methods match the Qiskit reference statevector.
Einsum probabilities: {'00': np.float64(0.5000000000000001), '11': np.float64(0.5000000000000001)}


## Try another circuit

You can replace the gates below and select any of the four methods with `Configuration(method=...)`.

In [9]:
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.x(1)
circuit.cx(0, 2)

result = qs.simulate(
    circuit,
    qs.Configuration(method="einsum", number_of_shots=shots),
)

print(result.statevector)
print(result.counts)

[4.32978028e-17+0.j 0.00000000e+00+0.j 7.07106781e-01+0.j
 0.00000000e+00+0.j 0.00000000e+00+0.j 4.32978028e-17+0.j
 0.00000000e+00+0.j 7.07106781e-01+0.j]
{'000': np.float64(1.8746997283273227e-33), '010': np.float64(0.5000000000000001), '101': np.float64(1.8746997283273227e-33), '111': np.float64(0.5000000000000001)}
